# Wikidata Airport Dataset — EDA

Exploratory analysis of `data/interim/wikidata_airports.csv`, the fused, conflict-resolved dataset produced by `wikidata_extraction.ipynb`. This is about the *final* dataset's shape, distributions, and anomalies — for how each attribute was sourced/profiled/resolved, see `wikidata_profiling.ipynb` and `wikidata_extraction.ipynb`.

In [43]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

df = pd.read_csv(project_root / "data" / "interim" / "wikidata_airports.csv")
df.shape

(24871, 13)

## Overview

In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24871 entries, 0 to 24870
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   airport     24871 non-null  str    
 1   name        24871 non-null  str    
 2   country     24855 non-null  str    
 3   elevation   18784 non-null  float64
 4   faa         11042 non-null  str    
 5   iata        7969 non-null   str    
 6   icao        11560 non-null  str    
 7   located_in  24502 non-null  str    
 8   operator    2871 non-null   str    
 9   owner       374 non-null    str    
 10  timezone    10190 non-null  str    
 11  lat         24820 non-null  float64
 12  lon         24820 non-null  float64
dtypes: float64(3), str(10)
memory usage: 2.5 MB


In [45]:
df.head()

,airport,name,country,elevation,faa,iata,icao,located_in,operator,owner,timezone,lat,lon
0,http://www.wikidata.org/entity/Q1000018,Ji'an Jinggangshan Airport,People's Republic of China,86.0000,NaN,JGS,ZSGS,Taihe County,NaN,NaN,NaN,26.857500,114.737222
1,http://www.wikidata.org/entity/Q1000025,Changde Taohuayuan Airport,People's Republic of China,NaN,NaN,CGD,ZGCD,Hunan,NaN,NaN,NaN,28.918900,111.640000
2,http://www.wikidata.org/entity/Q1002604,Jiujiang Lushan Airport,People's Republic of China,49.9872,NaN,JIU,ZSJJ,Jiujiang,NaN,NaN,NaN,29.476944,115.801111
3,http://www.wikidata.org/entity/Q1006478,Aksu Hongqipo Airport,People's Republic of China,1163.0000,NaN,AKU,ZWAK,Hongqipo,NaN,NaN,NaN,41.262500,80.291667
4,http://www.wikidata.org/entity/Q1006900,Mohe Gulian Airport,People's Republic of China,579.0000,NaN,OHE,ZYMH,Heilongjiang,NaN,NaN,NaN,52.921111,122.420556


## Missingness

In [46]:
(df.isna().mean().sort_values(ascending=False) * 100).round(2).rename("missing_%")

owner         98.50
operator      88.46
iata          67.96
timezone      59.03
faa           55.60
icao          53.52
elevation     24.47
located_in     1.48
lat            0.21
lon            0.21
country        0.06
airport        0.00
name           0.00
Name: missing_%, dtype: float64

In [47]:
# check whether any records have invalid coordinates
print(f"COUNT(lat = 0): {len(df[df['lat'] == 0])}")
print(f"COUNT(lon = 0): {len(df[df['lat'] == 0])}")

print(f"COUNT(lat > 90 or lat < -90): {len(df[(df['lat'] > 90) | (df['lat'] < -90)])}")
print(f"COUNT(lon > 180 or lon < -180): {len(df[(df['lon'] > 180) | (df['lon'] < -180)])}")


COUNT(lat = 0): 0
COUNT(lon = 0): 0
COUNT(lat > 90 or lat < -90): 0
COUNT(lon > 180 or lon < -180): 0


## Uniqueness / duplicate checks

One row per `airport` URI is expected by construction (it's the groupby key in the extraction notebook) — this checks whether that key is actually unique, and whether any (name, lat, lon) combination repeats, which would suggest the same real-world airport appearing under two different Wikidata items.

In [48]:
print("duplicate airport URIs:", df["airport"].duplicated().sum())
print("duplicate airport FAA:", df["faa"].dropna().duplicated().sum())
print("duplicate airport IATA:", df["iata"].dropna().duplicated().sum())
print("duplicate airport ICAO:", df["icao"].dropna().duplicated().sum())



duplicate airport URIs: 0
duplicate airport FAA: 8
duplicate airport IATA: 77
duplicate airport ICAO: 38


## Intra-source dedup

The naive `df["icao"].dropna().duplicated().sum()` above (and similarly for `iata`/`faa`) can't tell apart two different situations:
- **Different real airports that happen to share the same code** (data error, code reuse, or succession over time.
- **The same real airport recorded as two separate Wikidata items** (a genuine duplicate — e.g. Rafael Hernández Airport above, or Wattisham Airfield / RAF Wattisham).

We disambiguate using coordinate distance. For each duplicated code value,
- if the airports sharing it are far apart (> 1km), they're different entities and the code itself is untrustworthy for that airport (**Treatment A**: null it out). 
- If they're close together, they're likely the same entity recorded twice (**Treatment B**: merge into one row). Within a merge group, the row with the fewest missing fields becomes the `primary` (the others get coalesced into it and dropped) — a deliberate "most complete wins" rule, not just whichever row happened to come first in the original data.


Three groups (`EDDB`, `CO95`, `TKJ`) were manually reviewed and found to be false positives for Treatment B (real, distinct, nearby airports — not duplicates) — excluded from auto-merge on purpose.

In [49]:
import math

DISTANCE_THRESHOLD_KM = 1
KNOWN_FALSE_POSITIVES = {"EDDB", "CO95", "TKJ"}  # manually confirmed: not duplicates, don't auto-merge


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def classify_duplicates(df: pd.DataFrame, code_col: str):
    """For a code column, split its duplicated values into:
    - different_entity_values: max pairwise distance > threshold -> different real airports
    - same_entity_groups: max pairwise distance <= threshold -> likely the same airport, duplicated
    """
    sub = df.dropna(subset=[code_col, "lat", "lon"])
    dup_vals = sub[code_col][sub[code_col].duplicated(keep=False)].unique()
    different_entity_values, same_entity_groups = [], []
    for v in dup_vals:
        rows = sub[sub[code_col] == v]
        lats, lons = rows["lat"].tolist(), rows["lon"].tolist()
        maxd = max(
            haversine_km(lats[i], lons[i], lats[j], lons[j])
            for i in range(len(rows))
            for j in range(i + 1, len(rows))
        )
        if maxd > DISTANCE_THRESHOLD_KM:
            different_entity_values.append(v)
        else:
            same_entity_groups.append({"value": v, "airports": rows["airport"].tolist()})
    return different_entity_values, same_entity_groups


dup_results = {
    col: dict(zip(["different_entity_values", "same_entity_groups"], classify_duplicates(df, col)))
    for col in ["icao", "iata", "faa"]
}
for col, r in dup_results.items():
    print(f"{col}: {len(r['different_entity_values'])} values -> different entities (Treatment A), "
          f"{len(r['same_entity_groups'])} groups -> likely same entity (Treatment B)")

icao: 31 values -> different entities (Treatment A), 7 groups -> likely same entity (Treatment B)
iata: 72 values -> different entities (Treatment A), 4 groups -> likely same entity (Treatment B)
faa: 4 values -> different entities (Treatment A), 4 groups -> likely same entity (Treatment B)


In [50]:
# Treatment A: different entities sharing a code -> null out that code (untrustworthy)
df_clean = df.copy()
for col in ["icao", "iata", "faa"]:
    bad_values = dup_results[col]["different_entity_values"]
    mask = df_clean[col].isin(bad_values)
    df_clean.loc[mask, col] = None
    print(f"Treatment A - {col}: nulled {mask.sum()} values")

Treatment A - icao: nulled 62 values
Treatment A - iata: nulled 145 values
Treatment A - faa: nulled 8 values


In [51]:
def coalesce_fill(df: pd.DataFrame, primary_airport: str, duplicate_airport: str) -> None:
    """Fill in any blanks in `primary_airport`'s row using values from
    `duplicate_airport`'s row. Only fills gaps -- never overwrites a value
    `primary_airport` already has."""
    primary_row = df.index[df["airport"] == primary_airport][0]
    duplicate_row = df.index[df["airport"] == duplicate_airport][0]

    for column in df.columns:
        if column == "airport":
            continue
        primary_is_missing = pd.isna(df.at[primary_row, column])
        duplicate_has_value = pd.notna(df.at[duplicate_row, column])
        if primary_is_missing and duplicate_has_value:
            df.at[primary_row, column] = df.at[duplicate_row, column]


def count_missing_fields(df: pd.DataFrame, airport: str) -> int:
    """How many columns are null for this airport's row -- lower means more complete."""
    row = df[df["airport"] == airport].iloc[0]
    return int(row.isna().sum())


def find_duplicate_groups_to_merge() -> list[dict]:
    """Collect every same-entity group across icao/iata/faa, skipping known
    false positives. An airport can show up in more than one code's duplicate
    list (e.g. sharing both icao and iata with the same duplicate) -- once a
    pair has been claimed by one code, skip it for the others so we don't
    try to merge the same two rows twice. Within each group, the airport
    with the fewest missing fields (i.e. the most complete row) becomes the
    primary that the others get merged into."""
    already_claimed = set()
    groups_to_merge = []

    for code_column in ["icao", "iata", "faa"]:
        for group in dup_results[code_column]["same_entity_groups"]:
            if group["value"] in KNOWN_FALSE_POSITIVES:
                continue

            unclaimed_airports = [a for a in group["airports"] if a not in already_claimed]
            if len(unclaimed_airports) < 2:
                continue  # already merged via a different shared code

            unclaimed_airports.sort(key=lambda a: count_missing_fields(df_clean, a))
            primary_airport, *duplicate_airports = unclaimed_airports
            groups_to_merge.append(
                {
                    "code_column": code_column,
                    "code_value": group["value"],
                    "primary": primary_airport,
                    "duplicates": duplicate_airports,
                }
            )
            already_claimed.update(unclaimed_airports)

    return groups_to_merge


# --- Treatment B: same entity duplicated -> coalesce-merge into one row, drop the rest ---
merge_log = []
for group in find_duplicate_groups_to_merge():
    for duplicate_airport in group["duplicates"]:
        coalesce_fill(df_clean, group["primary"], duplicate_airport)
        merge_log.append(
            {
                "kept": group["primary"],
                "dropped": duplicate_airport,
                "code_type": group["code_column"],
                "code_value": group["code_value"],
            }
        )

dropped_airports = {entry["dropped"] for entry in merge_log}
df_clean = df_clean[~df_clean["airport"].isin(dropped_airports)].reset_index(drop=True)

print(f"Treatment B: merged {len(merge_log)} duplicate pairs, dropped {len(dropped_airports)} rows")
print(f"{len(df)} rows -> {len(df_clean)} rows")
pd.DataFrame(merge_log)

Treatment B: merged 12 duplicate pairs, dropped 12 rows
24871 rows -> 24859 rows


,kept,dropped,code_type,code_value
0,http://www.wikidata.org/entity/Q108415915,http://www.wikidata.org/entity/Q1432050,icao,WBGR
1,http://www.wikidata.org/entity/Q10850540,http://www.wikidata.org/entity/Q123469762,icao,ZUXJ
2,http://www.wikidata.org/entity/Q11824367,http://www.wikidata.org/entity/Q15781813,icao,MSSM
3,http://www.wikidata.org/entity/Q2977538,http://www.wikidata.org/entity/Q12692439,icao,GUXD
4,http://www.wikidata.org/entity/Q7275662,http://www.wikidata.org/entity/Q2542312,icao,EGUW
5,http://www.wikidata.org/entity/Q7278511,http://www.wikidata.org/entity/Q35309607,icao,AYTK
6,http://www.wikidata.org/entity/Q3756170,http://www.wikidata.org/entity/Q8060965,icao,ZLYL
7,http://www.wikidata.org/entity/Q117100760,http://www.wikidata.org/entity/Q111635252,iata,JSO
8,http://www.wikidata.org/entity/Q20456367,http://www.wikidata.org/entity/Q2901817,iata,JPR
9,http://www.wikidata.org/entity/Q49751959,http://www.wikidata.org/entity/Q107283569,faa,4OH6


In [52]:
out_path = project_root / "data" / "interim" / "wikidata_airports_deduped.csv"
df_clean.to_csv(out_path, index=False)
print(f"Saved {len(df_clean)} airports to {out_path}")

Saved 24859 airports to /Users/huanglinchun/Desktop/Master/Studies/Web-Data-Integration/Web-Data-Integration-Project/data/interim/wikidata_airports_deduped.csv


## Numeric attributes

`elevation` (metres) and `lat`/`lon` (decimal degrees). Coordinates should fall within [-90, 90] / [-180, 180] by construction (GeoSPARQL extraction), but elevation has no such hard bound — flagging physically implausible values (below the Dead Sea shore at ~-430 m, or above the highest paved airstrips at ~5,500 m) is a cheap sanity check for bad data.

In [53]:
df[["elevation", "lat", "lon"]].describe()

,elevation,lat,lon
count,18784.000000,24820.000000,24820.000000
mean,366.829505,29.379098,-48.788407
std,471.565076,23.702135,77.062083
min,-378.000000,-79.250000,-179.876945
25%,61.874400,26.170227,-96.394583
50%,223.723200,37.048462,-82.169461
75%,445.008000,43.116900,1.863542
max,4411.000000,82.517778,179.975500


In [54]:
implausible_elevation = df[(df["elevation"] < -430) | (df["elevation"] > 5500)]
print(f"{len(implausible_elevation)} airports with a physically implausible elevation")
implausible_elevation[["airport", "name", "elevation"]].sort_values("elevation", ascending=False).head(10)

0 airports with a physically implausible elevation


,airport,name,elevation


## Categorical / identifier attributes

In [55]:
df["country"].value_counts().head(15)

country
United States                       14023
Canada                               1407
Brazil                                667
Australia                             489
Russia                                393
Chile                                 353
People's Republic of China            338
Papua New Guinea                      289
Indonesia                             258
India                                 251
Democratic Republic of the Congo      210
United Kingdom                        209
France                                186
Argentina                             169
Bolivia                               168
Name: count, dtype: int64

In [56]:
for col in ["iata", "icao", "faa"]:
    print(f"{col}: {df[col].notna().sum()} airports have a value ({df[col].notna().mean():.1%})")

iata: 7969 airports have a value (32.0%)
icao: 11560 airports have a value (46.5%)
faa: 11042 airports have a value (44.4%)
